# Baseline

In [18]:
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(MODEL_NAME, max_seq_length=1024, load_in_4bit=False)

==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 8. Max memory: 79.097 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

unsloth/Qwen3-4B-Instruct-2507 does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [19]:
FastLanguageModel.for_inference(model)

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560, padding_idx=151669)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layerno

In [51]:
def generate_response(messages, model, tokenizer):
    # generate response
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt", padding=True, padding_side="left").to(model.device)

    # Generate
    outputs = model.generate(
        **inputs,
        max_new_tokens=1024,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

    responses = tokenizer.decode(outputs[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return responses


msg = [{'role': 'user', 'content': 'say hello'}]
messages = [msg, msg]
print(generate_response(messages, model, tokenizer))


['Hello! 🌟 How can I assist you today? 😊', 'Hello! 😊 How can I assist you today?']


## IF-Eval

In [29]:
from instruction_following_eval import get_examples, evaluate_instruction_following
from tqdm import tqdm

In [30]:
examples = get_examples()
examples[0]

{'key': 1000,
 'prompt': 'Write a 300+ word summary of the wikipedia page "https://en.wikipedia.org/wiki/Raymond_III,_Count_of_Tripoli". Do not use any commas and highlight at least 3 sections that has titles in markdown format, for example *highlighted section part 1*, *highlighted section part 2*, *highlighted section part 3*.',
 'instruction_id_list': ['punctuation:no_comma',
  'detectable_format:number_highlighted_sections',
  'length_constraints:number_words'],
 'kwargs': [{},
  {'num_highlights': 3},
  {'relation': 'at least', 'num_words': 300}]}

In [55]:
BATCH_SIZE = 16
responses = []
for i in tqdm(range(0, len(examples), BATCH_SIZE)):
    batch = examples[i:i+BATCH_SIZE]
    _msgs = [[{'role': 'user', 'content': example['prompt']}] for example in batch]
    responses.extend(generate_response(_msgs, model, tokenizer))

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 34/34 [13:12<00:00, 23.31s/it]


In [57]:
metrics = evaluate_instruction_following(examples, responses)
metrics


{'prompt_level_strict_accuracy': 0.822550831792976,
 'inst_level_strict_accuracy': 0.8752997601918465,
 'prompt_level_loose_accuracy': 0.8558225508317929,
 'inst_level_loose_accuracy': 0.8968824940047961}